
# Fine tuned Smolvlm-256M

##Install library

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes
!pip install peft datasets opencv-python-headless
!pip install trl
!pip install mlflow dagshub
!pip install jiwer rapidfuzz



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.6 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
   

##Load the model

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForImageTextToText, AutoProcessor

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
# Load model and tokenizer
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
    _attn_implementation="sdpa" # Use `flash_attention_2` on Ampere GPUs and above and `eager` on older GPUs.
)
processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.


In [ ]:
from peft import get_peft_model, LoraConfig

peft_config = LoraConfig(
    r=16, # Augmenté de 8 à 16 pour mieux capturer les détails
    lora_alpha=32,
    lora_dropout=0.05,
    # On cible plus de modules pour une meilleure "intelligence" sur les reçus
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    use_dora=False,
    init_lora_weights="gaussian"
)


# Apply PEFT model adaptation
peft_model = get_peft_model(
    model,
    peft_config
)

In [ ]:
# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 2,727,936 || all params: 259,212,864 || trainable%: 1.0524


##Preparation of data

In [ ]:
# Configuration
image_folder = "/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/"
json_folder = "/kaggle/input/datasets/ahmadoubg/dataset-receipt/dataJSON/"
output_path = "dataset.jsonl"


In [ ]:
import os
import json
from pathlib import Path

def build_dataset(image_folder, json_folder, output_path):

  prompt = (
          "Extract company, date, address and total from this receipt.\n"
          "If missing return null.\n"
          "Do NOT guess.\n"
          "Return ONLY valid JSON."
  )

  with open(output_path, "w") as out_file:
      for img_file in os.listdir(image_folder):
          if not img_file.endswith(".jpg"):
              continue

          base_name = Path(img_file).stem
          json_path = os.path.join(json_folder, base_name + ".json")

          if not os.path.exists(json_path):
              continue

          with open(json_path, "r") as f:
              data = json.load(f)

          # Safe JSON structure
          target = {
              "company": data.get("company", None),
              "date": data.get("date", None),
              "address": data.get("address", None),
              "total": data.get("total", None)
          }

          sample = {
              "img_path": os.path.join(image_folder, img_file),
              "messages": [
                  {
                      "role": "user",
                      "content": [
                          {"type": "image", "image": os.path.join(image_folder, img_file)},
                          {"type": "text", "text": prompt}
                      ]
                  },
                  {
                      "role": "assistant",
                      "content": [
                          {"type": "text", "text": json.dumps(target)}
                      ]
                  }
              ]
          }

          return out_file.write(json.dumps(sample) + "\n")

  print("✅ Dataset ready (jsonl)")

Split the data

In [ ]:
from datasets import load_dataset
data = build_dataset(image_folder, json_folder, output_path)
dataset = load_dataset("json", data_files= data)["train"]

✅ Dataset ready (jsonl)


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print("Original dataset[0] structure:")
print(dataset[0])

Original dataset[0] structure:
{'img_path': '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/623.jpg', 'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/623.jpg'}, {'type': 'text', 'text': 'Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '{"company": "B & BEST RESTAURANT", "date": "22/04/2017", "address": "NO.12,JALAN SS4C/5,PETALING JAYA SELANGOR DARUL EHSAN", "total": "22.25"}'}]}]}


In [ ]:
len(dataset)

950

### Preprocess and structure data for fine-tuning

In [ ]:
def process_metadata(example):
    prompt = "Extract company, date, address and total from this image. If missing return null. Do NOT guess. Answer ONLY in JSON."

    full_chat = example["messages"]

    prompt_chat = [example["messages"][0]]

    full_text = processor.apply_chat_template(
        full_chat,
        tokenize=False
    )

    prompt_text = processor.apply_chat_template(
        prompt_chat,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_len = len(processor.tokenizer(prompt_text)["input_ids"])

    return {
        "img_path": example["img_path"],
        "full_text": full_text,
        "prompt_len": prompt_len
    }

In [ ]:
dataset = dataset.map(
    process_metadata,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

### Convert a list of sample(batch) to a tensor format for training

In [ ]:
from PIL import Image
import torch

def collate_fn(batch):
    valid_images = []
    valid_texts = []
    valid_prompt_lens = []

    # ---------------------------
    # 1. Safe loading (no misalignment)
    # ---------------------------
    for item in batch:
        try:
            image = Image.open(item["img_path"]).convert("RGB")
            valid_images.append(image)
            valid_texts.append(item["full_text"])
            valid_prompt_lens.append(item["prompt_len"])
        except Exception as e:
            print(f"[WARNING] Failed image: {item.get('img_path')} -> {e}")

    if len(valid_images) == 0:
        raise ValueError("All images failed to load.")

    # ---------------------------
    # 2. Multimodal processing
    # ---------------------------
    inputs = processor(
        images=valid_images,
        text=valid_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024  # IMPORTANT: control truncation
    )

    input_ids = inputs["input_ids"]
    labels = input_ids.clone()

    # ---------------------------
    # 3. Mask padding tokens
    # ---------------------------
    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100

    # ---------------------------
    # 4. SAFE prompt masking
    # ---------------------------
    seq_len = labels.shape[1]

    for i, prompt_len in enumerate(valid_prompt_lens):
        # Prevent overflow if truncated
        safe_len = min(prompt_len, seq_len)

        # Mask prompt tokens
        labels[i, :safe_len] = -100

    # ---------------------------
    # 5. Attach labels
    # ---------------------------
    inputs["labels"] = labels

    return inputs

## Evaluation

### evaluation using metrics:

**Accuracy, Recall, Precision, F1 score and CER**

### Configuration for evaluation

In [ ]:
import re
import numpy as np
from jiwer import cer
from rapidfuzz import fuzz

# ---------------------------
# Config
# ---------------------------
FIELDS = ["company", "date", "address", "total"]

WEIGHTS = {
    "company": 1.0,
    "date": 2.0,
    "address": 1.0,
    "total": 3.0
}

FUZZY_FIELDS = ["company", "address"]
EXACT_FIELDS = ["date", "total"]

### Fonctions helpers

In [ ]:
def safe_json_extract(text):
    """Extract JSON safely from model output"""
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return None


def normalize_text(x):
    if x is None:
        return ""
    return str(x).strip().lower()


def is_valid_total(x):
    return bool(re.match(r'^\d+(\.\d{1,2})?$', x))


def is_valid_date(x):
    return bool(re.match(r'^\d{2}[/-]\d{2}[/-]\d{2,4}$', x))


def fuzzy_match(a, b, threshold=85):
    return fuzz.ratio(a, b) >= threshold

### Fonction eval

In [ ]:
def evaluate_sample(gt_str, pred_str):
    # CER global (optional)
    global_cer = cer(gt_str.lower(), pred_str.lower())

    gt_dict = safe_json_extract(gt_str)
    pred_dict = safe_json_extract(pred_str)

    if gt_dict is None or pred_dict is None:
        return {
            "valid_json": 0,
            "accuracy": 0,
            "precision": 0,
            "recall": 0,
            "f1": 0,
            "cer": global_cer,
            "field_cer": 0 # Added to prevent KeyError
        }

    tp, fp, fn = 0, 0, 0
    weighted_correct = 0
    total_weight = sum(WEIGHTS.values())

    field_cers = []

    for field in FIELDS:
        gt_v = normalize_text(gt_dict.get(field, ""))
        pred_v = normalize_text(pred_dict.get(field, ""))

        # --- CER per field ---
        if gt_v and pred_v:
            field_cers.append(cer(gt_v, pred_v))

        match = False

        # --- Matching logic ---
        if field in FUZZY_FIELDS:
            match = fuzzy_match(gt_v, pred_v)

        elif field in EXACT_FIELDS:
            if field == "total":
                if is_valid_total(pred_v):
                    match = (gt_v == pred_v)

            elif field == "date":
                if is_valid_date(pred_v):
                    match = (gt_v == pred_v)

        # --- Metrics update ---
        if gt_v:
            if match:
                tp += 1
                weighted_correct += WEIGHTS[field]
            elif not pred_v:
                fn += 1
            else:
                fp += 1
        else:
            if pred_v:
                fp += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    weighted_accuracy = weighted_correct / total_weight
    avg_field_cer = np.mean(field_cers) if field_cers else 0

    return {
        "valid_json": 1,
        "accuracy": weighted_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "cer": global_cer,
        "field_cer": avg_field_cer
     }

### Initialistion of dagshub and mflow for save experiments

In [ ]:
import dagshub
import mlflow

# Remplace par tes vrais identifiants DagsHub
dagshub.init(repo_owner="AhmadouBG", repo_name="smolvlm-receipts", mlflow=True)
mlflow.set_experiment("SmolVLM_Evaluation")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b871ccf7-e82f-46db-bf95-59c77de5a56f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5bdb721841c50a76db1792284d65717775f4dd789428a258d5569fa90c2c3400




Accessing as AhmadouBG

Initialized MLflow to track repo "AhmadouBG/smolvlm-receipts"

Repository AhmadouBG/smolvlm-receipts initialized!

<Experiment: artifact_location='mlflow-artifacts:/efb26ce8698d413b94f6c81be8257280', creation_time=1778109960292, experiment_id='0', last_update_time=1778109960292, lifecycle_stage='active', name='SmolVLM_Evaluation', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

Before fine tuning, let's check the first example for the model output using 20 sample and 10 sample after fine tuned with unseen data.

In [ ]:
import io
from PIL import Image
import base64

def run_and_log_evaluation(model, dataset_subset, run_name="Evaluation"):
    model.eval()
    all_metrics = []
    json_invalid_count = 0

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "model_type": "SmolVLM-256M",
            "temperature": 0.0001,
            "status": "post-finetuning" if hasattr(model, "peft_config") else "base"
        })

        for i in range(min(len(dataset_subset), 10)): # Augmenté à 50 pour plus de fiabilité
            example = dataset_subset[i]
            image_pil = Image.open(example['img_path']).convert("RGB")

            # Template propre
            input_text = processor.apply_chat_template([
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
            ], tokenize=False, add_generation_prompt=True)

            inputs = processor(images=image_pil, text=input_text, return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0001, do_sample=False)

            # Nettoyage strict
            generated_response = processor.decode(outputs[0], skip_special_tokens=True)
            # On ne garde que ce qui vient APRÈS le prompt
            generated_response = generated_response.split("Assistant:")[-1].strip()
            print("generated_response", generated_response)

            # Extraction de la réponse JSON
            # Extraction GT
            gt_json_str = example['full_text'].split("Assistant:")[-1].replace("<end_of_utterance>", "").strip()

            # Évaluation avec sécurité
            try:
                res = evaluate_sample(gt_json_str, generated_response)
                if not res.get('valid_json', False): json_invalid_count += 1
                all_metrics.append(res)
            except Exception as e:
                print(f"Erreur évaluation index {i}: {e}")

            image_pil.close() # Libérer la RAM

        # Métriques agrégées
        metrics_to_log = {
            "avg_precision": np.mean([m['precision'] for m in all_metrics]),
            "avg_recall": np.mean([m['recall'] for m in all_metrics]),
            "avg_f1": np.mean([m['f1'] for m in all_metrics]),
            "avg_cer": np.mean([m['cer'] for m in all_metrics]),
            "json_failure_rate": json_invalid_count / len(all_metrics) if all_metrics else 1
        }

        mlflow.log_metrics(metrics_to_log)
        print(f"✅ Terminé. F1: {metrics_to_log['avg_f1']:.2f} | JSON Fail: {metrics_to_log['json_failure_rate']:.2%} | CER: {metrics_to_log['avg_cer']:.2f} | Precision: {metrics_to_log['avg_precision']:.2f} | Recall: {metrics_to_log['avg_recall']:.2f}")


In [ ]:
run_and_log_evaluation(model, dataset, run_name="Base_Model_Test")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generated_response {
    "menu": {
        "nm": "SSTC/5",
        "num": "SSTC/5",
        "num": "PETALING JAYA",
        "price": "001808391968",
        "price_invoice": "21.00"
    },
    "sub_total": {
        "subtotal_price": [
            {
                "subtotal_price": "21.00",
                "tax_price": "-0.01"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "21.00"
            },
            {
                "subtotal_price": "2
generated_response {
    "menu": {
        "menu_id": "53300 kuala Lumpur",
        "menu_price": "53300.00

Configuration of image in the processor

In [ ]:
# Permet au modèle de voir plus de détails sur les tickets longs
processor.image_processor.do_resize = True
processor.image_processor.size = {"longest_edge": 768} # Teste 768 pour ton GPU Kaggle


Split data

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

Log of hugging face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('checkpoint')
login(token=token)
# Colab vous demandera votre jeton (token) "Write"


### Supervised Fine-tuning (SFT)

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    num_train_epochs=6,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    weight_decay=0.01,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,

    optim="adamw_torch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=3407,


    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=2048,

    report_to="none",
    output_dir="./SmolVLM-256M-Instruct-FineTuned-Receipt",

     # --- AJOUTS POUR HUGGING FACE ---
    push_to_hub=True,               # Active l'envoi automatique
    hub_model_id="gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1", # Nom sur le Hub
    hub_strategy="checkpoint",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collate_fn,
    peft_config=peft_config
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

Step,Training Loss,Validation Loss
100,0.609566,0.544946
200,0.220527,0.192336
300,0.127608,0.153863
400,0.117685,0.140516


### Recover the training checkpoint after the lost of session.

In [ ]:
from huggingface_hub import HfApi, RepoFile, RepoFolder

hf_api = HfApi()
repo_id = "gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1"

print(f"Listing contents of Hugging Face repository: {repo_id}")

try:
    repo_files = hf_api.list_repo_tree(repo_id=repo_id, recursive=True)

    checkpoint_found = False
    for item in repo_files:
        # Only consider RepoFolder items for directories
        if isinstance(item, RepoFolder):
            if 'checkpoint' in item.path:
                print(f"Found checkpoint directory: {item.path} (RepoFolder)")
                checkpoint_found = True

    if not checkpoint_found:
        print("No checkpoint directories found in the repository.")
    else:
        print("Checkpoints successfully found in the repository.")

except Exception as e:
    print(f"Error accessing Hugging Face repository {repo_id}: {e}")
    print("Please ensure your Hugging Face token has write access and is correctly logged in (cell `v2yutATEMgxl`).")

Listing contents of Hugging Face repository: gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1
Found checkpoint directory: last-checkpoint (RepoFolder)
Checkpoints successfully found in the repository.


In [ ]:
from huggingface_hub import snapshot_download
import os

# Define the Hugging Face repo ID where the checkpoints are stored
hf_checkpoint_repo_id = "gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1"

# Define the local directory where the trainer expects to find the checkpoints
local_checkpoint_dir = "./SmolVLM-256M-Instruct-FineTuned-Receipt"

# Create the directory if it doesn't exist
os.makedirs(local_checkpoint_dir, exist_ok=True)

print(f"Downloading checkpoint from {hf_checkpoint_repo_id} to {local_checkpoint_dir}...")

# Download the entire repository contents to the local_checkpoint_dir
# This will include the 'last-checkpoint' folder if it exists in the repo
snapshot_download(repo_id=hf_checkpoint_repo_id, local_dir=local_checkpoint_dir)

print("✅ Checkpoint download complete!")

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

✅ Checkpoint download complete!


In [ ]:
trainer.train(resume_from_checkpoint="./SmolVLM-256M-Instruct-FineTuned-Receipt/last-checkpoint")

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

Step,Training Loss,Validation Loss
600,0.120238,0.130441
642,0.111971,0.129905


TrainOutput(global_step=642, training_loss=0.02600235601080541, metrics={'train_runtime': 2569.1479, 'train_samples_per_second': 1.997, 'train_steps_per_second': 0.25, 'total_flos': 3047012379227520.0, 'train_loss': 0.02600235601080541})

In [ ]:
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained("./merged-model-full")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Load and merged the fine-tuned model

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
import gc

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()

# Define model IDs and paths
original_base_model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
peft_adapters_repo_id = "gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1"
merged_fp16_local_path = "./smolvlm_merged_fp16_for_gguf"

print("\n--- Creating full-precision merged model for GGUF conversion ---")

# 1. Load the original base model in full float16 precision (NO quantization)
print(f"Loading base model {original_base_model_id} in float16...")
base_model_fp16 = AutoModelForImageTextToText.from_pretrained(
    original_base_model_id,
    device_map="cuda", # Load to CPU to save GPU memory for merging
    torch_dtype=torch.float16, # Ensure full precision
    _attn_implementation="sdpa"
)

# 2. Load the processor (it's the same for base and PEFT models)
processor_fp16 = AutoProcessor.from_pretrained(original_base_model_id)

# 3. Load the PEFT adapters onto the full-precision base model
print(f"Loading PEFT adapters from {peft_adapters_repo_id}...")
peft_model_fp16 = PeftModel.from_pretrained(base_model_fp16, peft_adapters_repo_id)

# 4. Merge the adapters into the base model
print("Merging PEFT adapters into the base model (full precision)...")
merged_model_fp16 = peft_model_fp16.merge_and_unload()

# 5. Save the full-precision merged model and processor locally
print(f"Saving full-precision merged model to {merged_fp16_local_path}...")
merged_model_fp16.save_pretrained(merged_fp16_local_path)
processor_fp16.save_pretrained(merged_fp16_local_path)

print("✅ Full-precision merged model saved locally for GGUF conversion!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



--- Creating full-precision merged model for GGUF conversion ---
Loading base model HuggingFaceTB/SmolVLM-256M-Instruct in float16...


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

Loading PEFT adapters from gueye07/SmolVLM-Receipt-FineTune-No-Unslothv1...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/11.0M [00:00<?, ?B/s]

Merging PEFT adapters into the base model (full precision)...
Saving full-precision merged model to ./smolvlm_merged_fp16_for_gguf...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Full-precision merged model saved locally for GGUF conversion!


### Test with unseen data

In [ ]:
image_folder = "/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/"
json_folder = "/kaggle/input/datasets/ahmadoubg/dataset-unseen/dto/"
output_path = "dataset_test.jsonl"

In [ ]:

from datasets import load_dataset
data_test = build_dataset(image_folder, json_folder, output_path)
dataset_test = load_dataset("json", data_files = data_test)['test']
len(dataset_test)

✅ Dataset ready (jsonl)


Generating test split: 0 examples [00:00, ? examples/s]

17

In [ ]:
dataset_test['test'][0]

{'img_path': '/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/950.jpg',
 'messages': [{'role': 'user',
   'content': [{'type': 'image',
     'image': '/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/950.jpg'},
    {'type': 'text',
     'text': 'Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON.'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': '{"company": "KEDAI PAPAN YEW CHUAN", "date": "11/05/2018", "address": "LOT 276 JALAN BANTING 43800 DENGKIL, SELANGOR", "total": "68.90"}'}]}]}

In [ ]:
dataset_test_processed = dataset_test.map(
    process_metadata,
    remove_columns=dataset_test.column_names # Replace original columns with new ones from process_metadata
)

run_and_log_evaluation(merged_model_fp16, dataset_test_processed, run_name="FineTuned_Model_Test")

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

generated_response {
    "company": "KEDAI PAPAN YEW CHUAN",
    "date": "11/05/2018",
    "address": "LOT 276 JALAN BANTING",
    "total": "68.90"
}
generated_response {
    "company": "M.A.S.H. DISTRIBUTOR & MARKETING SDN BHD",
    "date": "20/06/2018",
    "address": "10400 PENANG",
    "total": "85.20"
}
generated_response {
    "clg": "GL HANDICRAFT & TAILORING",
    "gst": "1000194632736",
    "company": "MALAYSIA",
    "orderprice": "R 100.00",
    "totalprice": "R 100.00"
}
generated_response {
    "company": "ONE THREE SEAFOOD RESTAURANT SDN BHD",
    "date": "20-06-2018",
    "address": "43800 DENGKIL, JALAN AIR HITAM",
    "total": "38.00"
}
generated_response { "company": "OCEAN L.C. PACKAGING ENTERPRISE", "date": "27/06/2018", "address": "OCEAN L.C. PACKAGING ENTERPRISE", "total": "12.45"}
generated_response {
    "company": "PASAR RAYO MEGA MAJU",
    "date": "NO 24, 25 & 26, JALAN PASAT PERNIAGAAN BUNGA RAYA 4, PASAT PERNIAGAAN BUNGA RAYA, TEL: 01-283886",
    "price": "

## Convert the Merged Model to GGUF Format

Now that the fully merged model (base + LoRA adapters) is on the Hugging Face Hub, we can use the `llama.cpp` conversion script to convert it into the GGUF format. We'll generate two versions: F16 (full precision) and Q8_0 (8-bit quantized) for optimal performance and size balance.

## Install llama.cpp for convertion in gguf model

In [ ]:
!apt-get update
!apt-get install pciutils build-essential cmake curl libcurl4-openssl-dev -y
!git clone https://github.com/ggml-org/llama.cpp

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:13 https://cloud.r-project.org/bin/linux/u

In [ ]:
%cd llama.cpp
!cmake -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF # Changed DGGML_CUDA=ON to OFF to avoid CUDA linking error
!cmake --build build --config Release -j 12 --clean-first --target llama-quantize llama-cli llama-mtmd-cli llama-server llama-gguf-split
!cp build/bin/llama-* .
!ls -l .

/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU b

### Convert in gguf in f16

In [ ]:
%cd /kaggle/working/

# Define the local path for the full-precision merged model
merged_fp16_local_path = "./smolvlm_merged_fp16_for_gguf"

# Convert to F16 GGUF
print(f"Converting full-precision merged model to F16 GGUF from {merged_fp16_local_path}...")
!python llama.cpp/convert_hf_to_gguf.py "{merged_fp16_local_path}" \
    --outfile smolvlm_receipt_merged_f16.gguf --outtype f16
print("✅ F16 GGUF conversion complete!")

/kaggle/working
Converting full-precision merged model to F16 GGUF from ./smolvlm_merged_fp16_for_gguf...
INFO:hf-to-gguf:Loading model: smolvlm_merged_fp16_for_gguf
INFO:hf-to-gguf:Model architecture: VLlama3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {576, 49280}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {576, 49280}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {1536, 576}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {576}
INFO:hf

In [ ]:
%cd /kaggle/working/

# Define the Hugging Face repo ID for the merged model
merged_hf_repo_id = "gueye07/SmolVLM-256M-Instruct-FineTuned-Merged"

# Convert to F16 GGUF
print(f"Converting {merged_hf_repo_id} to F16 GGUF...")
!python llama.cpp/convert_hf_to_gguf.py "{merged_hf_repo_id}" \
    --outfile smolvlm_receipt_merged_f16.gguf --outtype f16
print("✅ F16 GGUF conversion complete!")

#### Convert in gguf in f16 immproj model

In [ ]:
%cd /kaggle/working/

# Define the local path for the full-precision merged model
merged_fp16_local_path = "./smolvlm_merged_fp16_for_gguf"

# Convert to F16 GGUF with mmproj
print(f"Converting full-precision merged model to F16 GGUF with mmproj from {merged_fp16_local_path}...")
!python llama.cpp/convert_hf_to_gguf.py "{merged_fp16_local_path}" \
    --outfile mmproj_smolvlm_receipt_merged_f16.gguf --outtype f16 --mmproj
print("✅ F16 mmproj GGUF conversion complete!")

/kaggle/working
Converting full-precision merged model to F16 GGUF with mmproj from ./smolvlm_merged_fp16_for_gguf...
INFO:hf-to-gguf:Loading model: smolvlm_merged_fp16_for_gguf
INFO:hf-to-gguf:Model architecture: Idefics3ForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:mm.model.fc.weight,                   torch.float16 --> F16, shape = {12288, 576}
INFO:hf-to-gguf:v.patch_embd.bias,                    torch.float16 --> F32, shape = {768}
INFO:hf-to-gguf:v.patch_embd.weight,                  torch.float16 --> F32, shape = {16, 16, 3, 768}
INFO:hf-to-gguf:v.position_embd.weight,               torch.float16 --> F32, shape = {768, 1024}
INFO:hf-to-gguf:v.blk.0.ln1.bias,                     torch.float16 --> F32, shape = {768}
INFO:hf-to-gguf:v.blk.0.ln1.weight,                   torch.float16 --> F32, shape = {768}
INFO:hf-to-gguf:v

In [ ]:
%cd /kaggle/working/

# Define the local path for the full-precision merged model
merged_fp16_local_path = "./smolvlm_merged_fp16_for_gguf"

# Convert to Q8_0 GGUF (8-bit quantization)
print(f"Converting full-precision merged model to Q8_0 GGUF from {merged_fp16_local_path}...")
!python llama.cpp/convert_hf_to_gguf.py "{merged_fp16_local_path}" \
    --outfile smolvlm_receipt_merged_q8_0.gguf --outtype q8_0
print("✅ Q8_0 GGUF conversion complete!")

/kaggle/working
Converting full-precision merged model to Q8_0 GGUF from ./smolvlm_merged_fp16_for_gguf...
INFO:hf-to-gguf:Loading model: smolvlm_merged_fp16_for_gguf
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: VLlama3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> Q8_0, shape = {576, 49280}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {576, 49280}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {1536, 576}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_norm.


### Test the llama-cli



In [ ]:
!llama.cpp/llama-cli \
-m /kaggle/working/smolvlm_receipt_merged_f16.gguf \
--mmproj /kaggle/working/mmproj_smolvlm_receipt_merged_f16.gguf \
--image /kaggle/input/datasets/ahmadoubg/dataset-receipt/images/001.jpg \
-p "Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON."


Loading model... |-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b9042-e3e3f8e46
model      : smolvlm_receipt_merged_f16.gguf
modalities : text, vision

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern
  /image <file>       add an image file

Loaded media from '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/001.jpg'

> Extract company, date, address and total from this receipt.
If missing return null.
Do NOT guess.
Return ONLY valid JSON.

|-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-

In [ ]:
# The generated output from llama.cpp was:
generated_response_llama_cpp = """
 { "company": "JOB MARKETING SDN BHD", "date": "02/01/2019", "address": "OJC MARKETING SDN BHD ROOC NO 538358 H", "total": "170.00"
 }
"""

# Find the ground truth for image 002.jpg in the dataset
gt_json_str_002 = None
for item in dataset:
    if item['img_path'] == '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/001.jpg':
        # Extract the assistant's response (ground truth JSON)
        full_text = item['full_text']
        gt_json_start_idx = full_text.find("Assistant:")
        if gt_json_start_idx != -1:
            gt_json_str_with_suffix = full_text[gt_json_start_idx + len("Assistant:"):].strip()
            gt_json_str_002 = gt_json_str_with_suffix.replace("<end_of_utterance>", "").strip()
        break

if gt_json_str_002 is None:
    print("Ground truth for 002.jpg not found in dataset.")
else:
    print(f"Ground Truth for 002.jpg: {gt_json_str_002}")
    print(f"Generated Response for 002.jpg: {generated_response_llama_cpp}")

    # Evaluate the sample
    result_llama_cpp = evaluate_sample(gt_json_str_002, generated_response_llama_cpp)

    print("-" * 40)
    print("EVALUATION (llama.cpp output for 002.jpg):")
    print(f"JSON Valid     : {result_llama_cpp['valid_json']}")
    print(f"Accuracy       : {result_llama_cpp['accuracy']*100:.2f}%")
    print(f"Precision      : {result_llama_cpp['precision']:.4f}")
    print(f"Recall         : {result_llama_cpp['recall']:.4f}")
    print(f"F1 Score       : {result_llama_cpp['f1']:.4f}")
    print(f"CER (global)   : {result_llama_cpp['cer']:.4f}")
    print(f"CER (fields)   : {result_llama_cpp['field_cer']:.4f}")
    print("-" * 40)

Ground Truth for 002.jpg: {"company": "OJC MARKETING SDN BHD", "date": "02/01/2019", "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR", "total": "170.00"}
Generated Response for 002.jpg: 
 { "company": "JOB MARKETING SDN BHD", "date": "02/01/2019", "address": "OJC MARKETING SDN BHD ROOC NO 538358 H", "total": "170.00"
 }

----------------------------------------
EVALUATION (llama.cpp output for 002.jpg):
JSON Valid     : 1
Accuracy       : 85.71%
Precision      : 0.7500
Recall         : 1.0000
F1 Score       : 0.8571
CER (global)   : 0.3553
CER (fields)   : 0.2357
----------------------------------------
